# Constrained Tikhonov regularization implemented by (accelerated) forward-backward splitting
We consider the two-dimensional deconvolution problems to find a non-negative function f given data 
$$
    d \sim \mathrm{Pois}(h*f)
$$
with a non-negative convolution kernel $h$, and $\mathrm{Pois}$ denotes the element-wise Poisson distribution.

We explore the use of the semismooth Newton method to implement constrained Tikhonov regularization 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\| h*f-d\|^2_{L^2(w)} + \alpha \|f\|^2_{L^2}\right]
$$
with a weight $w = \frac{1}{\sqrt{d+1}}$. The regularization parameter $\alpha$ is chosen by the discrepancy principle.   

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mplib

from regpy.operators.convolution import GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import TikhonovRegularizationSetting, RegularizationSetting
from regpy.solvers.linear.semismoothNewton import SemismoothNewton_nonneg
from regpy.solvers.linear.proximal_gradient import ForwardBackwardSplitting, FISTA
from regpy.solvers.linear.primal_dual import PDHG
from regpy.hilbert import L2
from regpy.stoprules import DualityGapStopping, CountIterations
from regpy.functionals.numpy import TVUniformGridFcts

from test_images import fatcross_ring
from comparison_plot import comparison_plot

from numpy.linalg import norm 

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

### creating synthetic data perturbed by white noise

In [ ]:
grid, exact_sol = fatcross_ring(M=256,N=256,fac=1.)
r"""grid is the underlying UniformGridFcts vector space, and exact_sol the exact solution."""
a=0.1
conv =  GaussianBlur(grid,a,pad_amount=32)
r"""Convolution operator $f\mapsto h*f$ for the convolution kernel $h(x)=\exp(-|x|_2^2/a^2)$."""
data = conv(exact_sol)
noise = np.random.randn(*data.shape)
data += (0.05*norm(data)/norm(noise))*noise
"""Simulated measured data. The Poisson distribution occurs if photon count detectors are used."""
comparison_plot(grid,exact_sol,data,title_left='noisy measurement data')

## setting up Tikhonov functional

In [ ]:
tv_func = TVUniformGridFcts(grid)
alpha = 1e-3
n_iter = 100 
setting = TikhonovRegularizationSetting(op=conv, penalty=tv_func, data_fid = L2,
                                             data_fid_shift=data,
                                             regpar=alpha)

### FISTA

In [ ]:
fista = FISTA(setting)
stop_FISTA= CountIterations(max_iterations=n_iter)
fista.run(stoprule=stop_FISTA)
    
comparison_plot(grid,exact_sol,fista.x,title_left='FISTA reco')